<a href="https://colab.research.google.com/github/miriamamin1213-ux/Seed42_Models/blob/main/GPT2_CN_Hancock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:

import os
import random
import numpy as np
import pandas as pd
import json

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

from transformers import (
    GPT2Tokenizer,
    GPT2Model
)

# ------------------------------------------------------------
# Read Data
# ------------------------------------------------------------

import gdown
import json
import pandas as pd

# Clinical data
gdown.download(
    id="1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q",
    output="clinical_data.json",
    quiet=False
)

# Pathological data
gdown.download(
    id="1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F",
    output="pathological_data.json",
    quiet=False
)

with open("clinical_data.json", "r") as f:
    clinical = pd.DataFrame(json.load(f))

with open("pathological_data.json", "r") as f:
    pathology = pd.DataFrame(json.load(f))

    df = clinical.merge(
    pathology,
    on='patient_id',
    how='inner'
)

print(df.shape)

print("Clinical shape:", clinical.shape)
print("Pathology shape:", pathology.shape)

# ============================================================
# HPV Labels
# ============================================================

df = df[
    df['hpv_association_p16'].isin(
        ['positive', 'negative']
    )
].copy()

df['HPV'] = df['hpv_association_p16'].map({
    'negative': 0,
    'positive': 1
})

# ============================================================
# GPT2 Hancock Features
# ============================================================

features = [
    'age_at_initial_diagnosis',
    'sex',
    'smoking_status',

    'primary_tumor_site',

    'pT_stage',
    'pN_stage',

    'histologic_type',

    'number_of_positive_lymph_nodes',
    'number_of_resected_lymph_nodes',

    'perinodal_invasion',
    'lymphovascular_invasion_L',
    'vascular_invasion_V',
    'perineural_invasion_Pn',

    'resection_status',
    'infiltration_depth_in_mm',

    'first_treatment_intent',
    'first_treatment_modality',
    'days_to_first_treatment',

    'adjuvant_treatment_intent',
    'adjuvant_radiotherapy',
    'adjuvant_radiotherapy_modality',
    'adjuvant_systemic_therapy',
    'adjuvant_systemic_therapy_modality',
    'adjuvant_radiochemotherapy'
]

# ============================================================
# Missing Values
# ============================================================

for col in features:

    if df[col].dtype == object:

        df[col] = df[col].fillna(
            df[col].mode()[0]
        )

    else:

        df[col] = df[col].fillna(
            df[col].median()
        )

# ============================================================
# Narrative Function
# ============================================================


def patient_to_text(row):

    text = (
        f"This patient was diagnosed at age {row['age_at_initial_diagnosis']}. "
        f"Sex is {row['sex']}. "
        f"Smoking status is {row['smoking_status']}. "
        f"Primary tumour site is {row['primary_tumor_site']}. "
        f"Histologic type is {row['histologic_type']}. "
        f"Pathological T stage is {row['pT_stage']}. "
        f"Pathological N stage is {row['pN_stage']}. "
        f"The patient has "
        f"{row['number_of_positive_lymph_nodes']} positive lymph nodes and "
        f"{row['number_of_resected_lymph_nodes']} resected lymph nodes. "
        f"Perinodal invasion status is {row['perinodal_invasion']}. "
        f"Lymphovascular invasion status is {row['lymphovascular_invasion_L']}. "
        f"Vascular invasion status is {row['vascular_invasion_V']}. "
        f"Perineural invasion status is {row['perineural_invasion_Pn']}. "
        f"Resection status is {row['resection_status']}. "
        f"Infiltration depth is {row['infiltration_depth_in_mm']} mm. "
        f"First treatment intent was {row['first_treatment_intent']}. "
        f"First treatment modality was {row['first_treatment_modality']}. "
        f"Days to first treatment was {row['days_to_first_treatment']}. "
        f"Adjuvant treatment intent was {row['adjuvant_treatment_intent']}. "
        f"Adjuvant radiotherapy was {row['adjuvant_radiotherapy']}. "
        f"Adjuvant radiotherapy modality was {row['adjuvant_radiotherapy_modality']}. "
        f"Adjuvant systemic therapy was {row['adjuvant_systemic_therapy']}. "
        f"Adjuvant systemic therapy modality was {row['adjuvant_systemic_therapy_modality']}. "
        f"Adjuvant radiochemotherapy was {row['adjuvant_radiochemotherapy']}. "
        f"Predict whether the patient is HPV positive or HPV negative."
    )

    return text

# ============================================================
# Create GPT2 Text Corpus
# ============================================================

texts = df.apply(
    patient_to_text,
    axis=1
).tolist()

labels = df['HPV'].tolist()

print("Patients:", len(texts))
print(pd.Series(labels).value_counts())






# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


SEED = 42
seed_everything(SEED)


# ============================================================
# Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)




# ============================================================
# Train-test split
# ============================================================

train_texts, test_texts, y_train, y_test = (
    train_test_split(
        texts,
        labels,
        test_size=0.2,
        random_state=SEED,
        stratify=labels
    )
)

print("\nTrain class distribution:")
print(pd.Series(y_train).value_counts())

print("\nTest class distribution:")
print(pd.Series(y_test).value_counts())


# ============================================================
# Tokenizer
# ============================================================

MODEL_NAME = "openai-community/gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(
    MODEL_NAME
)

# GPT-2 has no default padding token
tokenizer.pad_token = tokenizer.eos_token

MAX_LENGTH = 256


# ============================================================
# Dataset
# ============================================================

class HPVTextDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_length=128
    ):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):

        encoding = self.tokenizer(
            self.texts[index],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": (
                encoding["input_ids"].squeeze(0)
            ),
            "attention_mask": (
                encoding["attention_mask"].squeeze(0)
            ),
            "label": torch.tensor(
                self.labels[index],
                dtype=torch.long
            )
        }


train_dataset = HPVTextDataset(
    texts=train_texts,
    labels=y_train,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

test_dataset = HPVTextDataset(
    texts=test_texts,
    labels=y_test,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)


# ============================================================
# DataLoaders
# ============================================================

BATCH_SIZE = 256

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


# ============================================================
# GPT-2 text classifier
# ============================================================

class HPVNetGPT2Text(nn.Module):

    def __init__(
        self,
        model_name="openai-community/gpt2",
        num_classes=2,
        dropout=0.1,
        freeze_gpt2=True
    ):
        super().__init__()

        self.freeze_gpt2 = freeze_gpt2

        self.gpt2 = GPT2Model.from_pretrained(
            model_name
        )

        self.gpt2.config.use_cache = False
        self.gpt2.config.pad_token_id = (
            tokenizer.pad_token_id
        )

        hidden_size = self.gpt2.config.hidden_size

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

        if freeze_gpt2:
            self.gpt2.requires_grad_(False)

    def train(self, mode=True):
        super().train(mode)

        if self.freeze_gpt2:
            self.gpt2.eval()

        return self

    def forward(
        self,
        input_ids,
        attention_mask
    ):
        outputs = self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True
        )

        hidden_states = outputs.last_hidden_state

        # Masked mean pooling
        expanded_mask = (
            attention_mask
            .unsqueeze(-1)
            .expand_as(hidden_states)
            .float()
        )

        hidden_sum = (
            hidden_states * expanded_mask
        ).sum(dim=1)

        valid_token_count = (
            expanded_mask.sum(dim=1)
            .clamp(min=1e-9)
        )

        pooled_output = (
            hidden_sum / valid_token_count
        )

        logits = self.classifier(
            pooled_output
        )

        return logits


# ============================================================
# Model
# ============================================================

model = HPVNetGPT2Text(
    model_name=MODEL_NAME,
    num_classes=2,
    freeze_gpt2=True
).to(device)


# ============================================================
# Class weights
# ============================================================

class_counts = np.bincount(y_train)

class_weights = (
    len(y_train)
    / (
        len(class_counts) *
        class_counts
    )
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

print("\nClass weights:", class_weights)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)


# ============================================================
# Optimiser
# ============================================================

optimizer = torch.optim.AdamW(
    filter(
        lambda parameter: parameter.requires_grad,
        model.parameters()
    ),
    lr=1e-3,
    weight_decay=1e-4
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
)


# ============================================================
# Training
# ============================================================

EPOCHS = 100
best_val_loss = float("inf")
best_epoch = 0

for epoch in range(EPOCHS):

    model.train()

    train_loss_sum = 0.0
    train_sample_count = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch[
            "attention_mask"
        ].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        train_loss = criterion(
            outputs,
            labels
        )

        train_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            filter(
                lambda parameter: parameter.requires_grad,
                model.parameters()
            ),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum += (
            train_loss.item() *
            input_ids.size(0)
        )

        train_sample_count += (
            input_ids.size(0)
        )

    average_train_loss = (
        train_loss_sum /
        train_sample_count
    )

    model.eval()

    test_loss_sum = 0.0
    test_sample_count = 0

    with torch.no_grad():

        for batch in test_loader:

            input_ids = batch[
                "input_ids"
            ].to(
                device,
                non_blocking=True
            )

            attention_mask = batch[
                "attention_mask"
            ].to(
                device,
                non_blocking=True
            )

            labels = batch["label"].to(
                device,
                non_blocking=True
            )

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            test_loss = criterion(
                outputs,
                labels
            )

            test_loss_sum += (
                test_loss.item() *
                input_ids.size(0)
            )

            test_sample_count += (
                input_ids.size(0)
            )

    average_test_loss = (
        test_loss_sum /
        test_sample_count
    )

    if average_test_loss < best_val_loss:

        best_val_loss = average_test_loss
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            "best_model_gpt2_text.pth"
        )

    if (epoch + 1) % 5 == 0:

        print(
            f"Epoch {epoch + 1:03d}, "
            f"Train={average_train_loss:.4f}, "
            f"Test={average_test_loss:.4f}, "
            f"Best Epoch={best_epoch}"
        )


print("\nBest Test Loss:", best_val_loss)
print("Best Epoch:", best_epoch)


# ============================================================
# Load best model
# ============================================================

checkpoint = torch.load(
    "best_model_gpt2_text.pth",
    map_location=device
)

model.load_state_dict(checkpoint)
model.eval()


# ============================================================
# Inference
# ============================================================

all_probabilities = []
all_predictions = []
all_targets = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch[
            "attention_mask"
        ].to(
            device,
            non_blocking=True
        )

        labels = batch["label"]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        all_probabilities.append(
            probabilities.cpu()
        )

        all_predictions.append(
            predictions.cpu()
        )

        all_targets.append(
            labels.cpu()

                    )


probabilities = torch.cat(
    all_probabilities,
    dim=0
).numpy()

predicted = torch.cat(
    all_predictions,
    dim=0
).numpy()

y_true = torch.cat(
    all_targets,
    dim=0
).numpy()


# ============================================================
# Evaluation
# ============================================================

results = classification_report(
    y_true,
    predicted,
    digits=4,
    zero_division=0
)

print("\nClassification report:")
print(results)

bal_acc = balanced_accuracy_score(
    y_true,
    predicted
)

f1 = f1_score(
    y_true,
    predicted,
    zero_division=0
)

auc = roc_auc_score(
    y_true,
    probabilities[:, 1]
)

print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"AUC:               {auc:.4f}")

Downloading...
From: https://drive.google.com/uc?id=1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q
To: /content/clinical_data.json
100%|██████████| 810k/810k [00:00<00:00, 84.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F
To: /content/pathological_data.json
100%|██████████| 458k/458k [00:00<00:00, 120MB/s]


(763, 49)
Clinical shape: (763, 32)
Pathology shape: (763, 18)
Patients: 332
0    191
1    141
Name: count, dtype: int64
Using device: cuda

Train class distribution:
0    152
1    113
Name: count, dtype: int64

Test class distribution:
0    39
1    28
Name: count, dtype: int64


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Class weights: tensor([0.8717, 1.1726], device='cuda:0')
Trainable parameters: 100226
Epoch 005, Train=0.7419, Test=0.7081, Best Epoch=5
Epoch 010, Train=0.6947, Test=0.6952, Best Epoch=9
Epoch 015, Train=0.6934, Test=0.6956, Best Epoch=9
Epoch 020, Train=0.6977, Test=0.6926, Best Epoch=19
Epoch 025, Train=0.6892, Test=0.6934, Best Epoch=22
Epoch 030, Train=0.6964, Test=0.6907, Best Epoch=30
Epoch 035, Train=0.6947, Test=0.6969, Best Epoch=30
Epoch 040, Train=0.6953, Test=0.6907, Best Epoch=38
Epoch 045, Train=0.6890, Test=0.6953, Best Epoch=41
Epoch 050, Train=0.6854, Test=0.6927, Best Epoch=48
Epoch 055, Train=0.6924, Test=0.6892, Best Epoch=55
Epoch 060, Train=0.6901, Test=0.6897, Best Epoch=55
Epoch 065, Train=0.6929, Test=0.6939, Best Epoch=55
Epoch 070, Train=0.6961, Test=0.6916, Best Epoch=67
Epoch 075, Train=0.6927, Test=0.6907, Best Epoch=67
Epoch 080, Train=0.6921, Test=0.6913, Best Epoch=78
Epoch 085, Train=0.6891, Test=0.6875, Best Epoch=85
Epoch 090, Train=0.6917, Test=0.